In [ ]:
import numpy as np 
import pandas as pd 
import os
import gc
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import optuna
from category_encoders import MEstimateEncoder, CatBoostEncoder, OrdinalEncoder
from sklearn import set_config
import category_encoders
from sklearn.decomposition import PCA
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.metrics import roc_auc_score, roc_curve, make_scorer, f1_score, accuracy_score
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer, KNNImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.preprocessing import FunctionTransformer,StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.preprocessing import PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, BaggingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import auc, roc_auc_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.gaussian_process import GaussianProcessClassifier
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier, Pool, cv
import warnings

warnings.filterwarnings("ignore", "use_inf_as_na")

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
sns.set_theme(style = 'white', palette = 'viridis')
pal = sns.color_palette('viridis')

pd.set_option('display.max_rows', 150)

# <b>Load Data</b>

In [ ]:
train = pd.read_csv(r'/kaggle/input/playground-series-s4e3/train.csv', index_col='id')
test = pd.read_csv(r'/kaggle/input/playground-series-s4e3/test.csv', index_col='id')
sub  = pd.read_csv(r'/kaggle/input/playground-series-s4e3/sample_submission.csv')

In [ ]:
train.head(3)

In [ ]:
test.head(3)

# <b>Dimension of data</b>

In [ ]:
print(f'Train: {train.shape}\nTest: {test.shape}')

# <b>Descriptive Statistic</b>

In [ ]:
def summary(wdf) -> pd.DataFrame():
    df = wdf.copy()
    desc = pd.DataFrame(index = list(df))
    desc['type'] = df.dtypes
    desc['count'] = df.count()
    desc['nunique'] = df.nunique()
    desc['%unique'] = desc['nunique'] /len(df) * 100
    desc['null'] = df.isnull().sum()
    desc['%null'] = desc['null'] / len(df) * 100
    desc = pd.concat([desc,df.describe().T.drop('count',axis=1)],axis=1)
    desc = desc.sort_values(by=['type','null'])
    
    return desc

In [ ]:
TARGETS = ['Pastry', 'Z_Scratch', 'K_Scatch', 'Stains', 'Dirtiness', 'Bumps', 'Other_Faults']

In [ ]:
desc_train = summary(train.drop(TARGETS,axis=1))
desc_train.style.background_gradient()

* <b>X_Minimum and X_Maximum</b>:
         Both variables appear to represent spatial coordinates or dimensions in a two-dimensional system.
         The minimum and maximum values indicate the range of values observed along the X axis.
         The mean and standard deviation give an idea of how dispersed the data is around the mean.
        

* <b>Y_Minimum and Y_Maximum:</b>
         Similar to X variables, but representing dimensions along the Y axis.
         The minimum and maximum values indicate the range of values observed along the Y axis.
         The mean and standard deviation indicate the dispersion of the data around the mean.
         

* <b>Pixels_Areas:</b>
         It appears to represent the area in pixels of some region or object.
         The minimum and maximum values indicate the variation in the observed areas.
         The mean and standard deviation show the mean and spread of area sizes.
     

* <b>X_Perimeter and Y_Perimeter:</b>
         They represent the perimeters along the X and Y axes, respectively.
         The minimum and maximum values indicate the variation in the observed perimeters.
         The mean and standard deviation give an idea of the dispersion of the perimeters around the mean.
     

 * <b>Sum_of_Luminosity, Minimum_of_Luminosity and Maximum_of_Luminosity:</b>
         They seem to be related to luminosity in some way.
         The minimum and maximum values indicate the variation in observed luminosity.
         The mean and standard deviation give an idea of how dispersed the data is around the mean.
      

* <b>Length_of_Conveyer:</b>
         It appears to represent the length of some conveyor belt or conveyor.
         The minimum and maximum values indicate the variation in observed lengths.
         The mean and standard deviation show the mean and spread of lengths.
       

 * <b>TypeOfSteel_A300 and TypeOfSteel_A400:</b>
         They appear to be binary variables indicating the type of steel.
         Statistics are not as relevant to these variables as they are binary.        


 * <b>Steel_Plate_Thickness:</b>
         Represents the thickness of steel plates.
         The minimum and maximum values indicate the variation in observed thicknesses.
         The mean and standard deviation give an idea of the dispersion of thicknesses around the mean.
    

* <b>Edges_Index, Empty_Index, Square_Index, Outside_X_Index, Edges_X_Index, Edges_Y_Index, Outside_Global_Index, LogOfAreas, Log_X_Index, Log_Y_Index, Orientation_Index, Luminosity_Index and SigmoidOfAreas: </b>

        They appear to be indices or measurements related to the shape, distribution or characteristics of some object or region.
         The minimum and maximum values indicate the variation in the observed indices.

         

In [ ]:
TARGETS = ['Pastry', 'Z_Scratch', 'K_Scatch', 'Stains', 'Dirtiness', 'Bumps', 'Other_Faults']
CAT_COLS = ['TypeOfSteel_A300','TypeOfSteel_A400','Outside_Global_Index']
NUM_COLS = [c for c in train.columns if c not in TARGETS and c not in CAT_COLS]

# <b>EDA</b>

# Numeric Features - Distribuition

In [ ]:
fig, ax = plt.subplots(6,4, figsize=(15,15), dpi=300) 
ax = ax.flatten()
for i, col in enumerate(NUM_COLS):
    sns.histplot(train[col],ax=ax[i],color='b',kde=True)
    sns.histplot(test[col],ax=ax[i],color='g',kde=True)    
    ax[i].set_title(f'{col}')
    ax[i].set_xlabel(None)    
    
for j in range(len(NUM_COLS),len(ax)):
    ax[j].axis('off')
fig.suptitle('Distribution of Feature\nper Dataset\n', fontsize = 24, fontweight = 'bold')
fig.legend(['Train', 'Test'])
plt.tight_layout(h_pad=0.1, w_pad=0.5)
plt.show()

# Categorical Features - Distribuition

In [ ]:
fig, ax = plt.subplots(3,1,figsize=(7,10))
ax = ax.flatten()
for i,c in enumerate(CAT_COLS):
    train[c].value_counts().plot(kind='bar',ax=ax[i],colormap='Paired')
    ax[i].bar_label(ax[i].containers[0], label_type='center')
plt.tight_layout(h_pad=0.1,w_pad=5)


* Outside_Global_index with only 1 value of 0.7, which could be a possible outlier. To do this, let's also check the test data.

In [ ]:
plot = test['Outside_Global_Index'].value_counts().plot(kind='bar',colormap='Paired')
plt.bar_label(plot.containers[0], label_type='center');

* As we can see, there is no 0.7 in the test data, so during modeling, the value of 0.7 will be replaced by 0.5.

# Target's

In [ ]:
ax = train[TARGETS].sum(axis=0).sort_values().plot(kind='bar')
ax.bar_label(ax.containers[0], label_type = 'edge');
plt.ylabel('Freq');

# <b>Adjust Target for models</b>

In [ ]:
def assign_class(row):
    for key, value in row.items():
        if value == 1:
            return key
    return 'Other_Faults'

train['target'] = train[TARGETS].apply(assign_class,axis=1)
le = LabelEncoder()
train['target'] = le.fit_transform(train['target'])

In [ ]:
train['target'].value_counts(normalize=True)*100

In [ ]:
train = train.drop(TARGETS,axis=1)

In [ ]:
train.head(2)

# <b>Correlation</b>

In [ ]:
fig, ax = plt.subplots(figsize=(25,25))
df1 = train._get_numeric_data()
corr = df1.corr(method='spearman')
mask = np.zeros_like(corr)
mask[np.triu_indices_from(mask)]= True
sns.heatmap(data=corr,mask=mask,annot=True,ax=ax);

In [ ]:
corr['target'].sort_values(ascending=False)[1:10]

In [ ]:
def distance(data, label = ''):
    corr = data.corr(method = 'spearman')
    dist_linkage = linkage(squareform(1 - abs(corr)), 'complete')
    
    plt.figure(figsize = (10, 8), dpi = 300)
    dendro = dendrogram(dist_linkage, labels=data.columns, leaf_rotation=90)
    plt.title(f'Feature Distance in {label} Dataset', weight = 'bold', size = 20)
    plt.show()

* For better visualization, we can use dendrogram

In [ ]:
distance(corr)

* <b>Pixels_Areas and X_Perimeter (0.878027)</b>: This strong positive correlation indicates that the size of the pixel areas is strongly related to the perimeter in the X direction. This suggests that as the pixel area increases, the perimeter in the X direction also tends to increase proportionally.

* <b>Pixels_Areas and Sum_of_Luminosity (0.966650)</b>: This very strong positive correlation suggests that pixel area is highly related to luminosity sum. This implies that larger regions in the image tend to have a higher luminosity sum.

* <b>Pixels_Areas and LogOfAreas (0.998439):</b> This almost perfect positive correlation indicates a highly linear relationship between the pixel area and the logarithm of the pixel area. This suggests that the size of the pixel areas grows exponentially as the pixel area increases.

* <b>Sum_of_Luminosity and LogOfAreas (0.967505)</b>: This strong positive correlation suggests a relationship between the sum of luminosity and the logarithm of the pixel area. This implies that the sum of luminosity can increase exponentially with the size of the pixel areas.

## Corelation of categorical features

* To calculate the brightness between categorical attributes, we use the association measure. And a measure of association that I will use, in this case, is Cramer's V.

In [ ]:
def cramer_v(var_x, var_y):
    confusion_matrix = pd.crosstab(var_x, var_y).values

    n = confusion_matrix.sum()
    r, k = confusion_matrix.shape

    chi2 = scipy.stats.chi2_contingency(confusion_matrix)[0]

    chi2corr = max(0, chi2 - (k-1) * (r-1) / (n-1))
    kcorr = k - (k-1) ** 2 / (n-1)
    rcorr = r - (r-1) ** 2 / (n-1)

    return np.sqrt((chi2corr/n) / min(kcorr-1, rcorr-1))


In [ ]:
a1 = cramer_v(train['TypeOfSteel_A300'], train['TypeOfSteel_A400'])
a2 = cramer_v(train['TypeOfSteel_A300'], train['Outside_Global_Index'])
a3 = cramer_v(train['TypeOfSteel_A300'], train['TypeOfSteel_A300'])

a4 = cramer_v(train['TypeOfSteel_A400'], train['TypeOfSteel_A400'])
a5 = cramer_v(train['TypeOfSteel_A400'], train['TypeOfSteel_A300'])
a6 = cramer_v(train['TypeOfSteel_A400'], train['Outside_Global_Index'])

a7 = cramer_v(train['Outside_Global_Index'], train['Outside_Global_Index'])
a8 = cramer_v(train['Outside_Global_Index'], train['TypeOfSteel_A400'])
a9 = cramer_v(train['Outside_Global_Index'], train['TypeOfSteel_A300'])

d = pd.DataFrame({'TypeOfSteel_A300': [a1, a2, a3],
              'TypeOfSteel_A400': [a4, a5, a6],
              'Outside_Global_Index': [a7, a8, a9],
             })

d = d.set_index(d.columns)
fig, ax = plt.subplots()
ax.figure.set_size_inches(16, 8)
sns.heatmap(d, annot=True, annot_kws={"fontsize":14})
plt.show()

* all correlations of categorical variables are weak

# <b>Preparation Model</b>

In [ ]:
def AdjustOutSideGlobalIndex(df):
    x_copy = df.copy()
    x_copy['Outside_Global_Index'] = np.where(x_copy['Outside_Global_Index']==0.7,0.5,x_copy['Outside_Global_Index'])
    return x_copy

TransformerOutSideGlobalIndex = FunctionTransformer(AdjustOutSideGlobalIndex)

In [ ]:
def LogFeatures(df):
    x_copy = df.copy()
    log_features = [f for f in NUM_COLS if (x_copy[f]>=0).all() and (scipy.stats.skew(x_copy[f])>0)]
    x_copy[log_features] = np.log1p(x_copy[log_features])
    return x_copy

TransformerLogFeatures = FunctionTransformer(LogFeatures)

In [ ]:
SEED = 42
cv = StratifiedKFold(n_splits=10, random_state=SEED,shuffle=True)

def cross_model_score(estimator, label = ''):
    
    X = train.copy()
    y = X.pop('target')
        
    val_predictions = np.zeros((len(train)))
    val_predictions = np.zeros((len(train)))
    train_scores, val_scores = [], []
    
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
    
        model = clone(estimator)
        
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]
    
        X_val = X.iloc[val_idx]
        y_val = y.iloc[val_idx]
        
        model.fit(X_train, y_train)
                
        train_preds = model.predict_proba(X_train)
        val_preds = model.predict_proba(X_val)
        
        train_score = roc_auc_score(y_train,train_preds,multi_class='ovr')
        val_score = roc_auc_score(y_val, val_preds,multi_class='ovr')        

        val_predictions[val_idx] = np.argmax(val_preds,axis=1)

        train_scores.append(train_score)
        val_scores.append(val_score)
    
    print(f'Val Score: {np.mean(val_scores):.5f} ± {np.std(val_scores):.5f} | Train Score: {np.mean(train_scores):.5f} ± {np.std(train_scores):.5f} | {label}')

    return val_scores, val_predictions, np.mean(val_scores)

# Models

In [ ]:
oof_list, score_list = pd.DataFrame(), pd.DataFrame()

# Tree Models

## ExtraTrees

* One of the most important parameters of ExtraTressClassifier is min_samples_leaf. A high value causes training to be faster but underfitting may occur, otherwise a low value may lead to slower training and overfitting may occur.

In [ ]:
scores = []
for max_iter in range(100, 400, 100):
    for min_samples_leaf in range(3, 8):        
        _,_,score = cross_model_score(ExtraTreesClassifier(random_state=SEED,
                                                           min_samples_leaf=min_samples_leaf,
                                                           n_estimators=max_iter))
        scores.append((max_iter,min_samples_leaf,score))
    
        

In [ ]:
max_iters = [score[0] for score in scores]
min_samples_leafs = [score[1] for score in scores]
scores_values = [score[2] for score in scores]

fig = plt.figure(figsize=(10,10))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(max_iters, min_samples_leafs, scores_values)

ax.set_xlabel('Max Iter')
ax.set_ylabel('Min Samples Leaf')
ax.set_zlabel('Score')

plt.show()

In [ ]:
best_index = scores_values.index(max(scores_values))

best_max_iter = max_iters[best_index]
best_min_samples_leaf = min_samples_leafs[best_index]
best_score = scores_values[best_index]

print(f'best combination:\nmax_iter: {best_max_iter} - min_samples_leaf: {best_min_samples_leaf} - score: {best_score}')

In [ ]:
score_list['ExtraTrees'], oof_list['ExtraTrees'],_ = cross_model_score(make_pipeline(TransformerOutSideGlobalIndex,                                                  
                                                      ExtraTreesClassifier(random_state=SEED,
                                                                           min_samples_leaf=4,
                                                                           n_estimators=300)))


## RandomForest

In [ ]:
scores_rf = []
for min_samples_leaf in range(2,20,1):
    _,_,scorerf = cross_model_score(
            make_pipeline(TransformerOutSideGlobalIndex,
                          RandomForestClassifier(n_estimators=100,
                          min_samples_leaf=min_samples_leaf,
                          random_state=SEED)))

    scores_rf.append((min_samples_leaf,scorerf))


In [ ]:
plt.figure(figsize=(6, 3))
plt.scatter([p for p, s in scores_rf],
            [s for p, s in scores_rf])
plt.xlabel('min samples leaf')
plt.ylabel('AUC score')
plt.title('min samples leaf vs AUC - Random Forest')
plt.xlim(2,20)

plt.show()

In [ ]:
best_score_rf = max([s for p, s in scores_rf])
best_index_rf = [s for p, s in scores_rf].index(best_score_rf)
print("Best result:", scores_rf[best_index_rf])

*  the best value for min_samples_leaf is 8

In [ ]:
score_list['RF'],oof_list['RF'],_ = cross_model_score(make_pipeline(TransformerOutSideGlobalIndex,                                                  
                                                      RandomForestClassifier(random_state=SEED,
                                                                             min_samples_leaf=8)))


## Logistic Regression

In [ ]:
score_list['LogisticRegression'],oof_list['LogisticRegression'],_ = cross_model_score(make_pipeline(TransformerOutSideGlobalIndex,
                                                  TransformerLogFeatures,
                                                  PolynomialFeatures(2),
                                                  StandardScaler(),
                                                  LogisticRegression(max_iter=100000)))


## XGBoost

In [ ]:
params_xgb = {'objective': 'multi:softmax',
             'num_class': 7,
             'tree_method': 'hist',
             'max_depth': 10,
             'learning_rate': 0.03513993699286939,
             'n_estimators': 858,
             'gamma': 2.518374355263247,
             'min_child_weight': 3,
             'colsample_bytree': 0.3354444954368183,
             'subsample': 0.7926423415125474}

score_list['XGB'],oof_list['XGB'],_ = cross_model_score(make_pipeline(TransformerOutSideGlobalIndex,                                                  
                                                        XGBClassifier(**params_xgb)))


## LightGBM


In [ ]:
params_lgbm = {'objective': 'multiclass',
               'num_class': 7,
               'n_estimators': 487,
               'learning_rate': 0.01769733167760597,
               'max_depth': 8,
               'reg_alpha': 1.3066859935623705,
               'reg_lambda': 5.254896833391085,
               'num_leaves': 19,
               'subsample': 0.31249582747908944,
               'colsample_bytree': 0.1941606712019325,
               'verbose': -1}
score_list['LGBM'],oof_list['LGBM'],_ = cross_model_score(make_pipeline(TransformerOutSideGlobalIndex,                                                  
                                                        LGBMClassifier(**params_lgbm)))


In [ ]:
ax = pd.DataFrame(score_list).mean().sort_values().plot(kind='barh')
ax.bar_label(ax.containers[0], label_type='center', color='white');

# <b>Submission</b>

In [ ]:
le.classes_

In [ ]:
model = make_pipeline(TransformerOutSideGlobalIndex,                                                  
            XGBClassifier(**params_xgb))
X = train.copy()
y = X.pop('target')
model.fit(X,y)
y_pred = model.predict_proba(test)

In [ ]:
sub['Pastry'] = y_pred[:, 4]
sub['Z_Scratch'] = y_pred[:, 6]
sub['K_Scatch'] = y_pred[:, 2]
sub['Stains'] = y_pred[:, 5]
sub['Dirtiness'] = y_pred[:, 1]
sub['Bumps'] = y_pred[:, 0]
sub['Other_Faults'] = y_pred[:, 3]

sub.head()